In [5]:
import os
import sys

repo_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "part1" else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from part1 import (
    gaussian_eliminate,
    back_substitution,
    determinant,
    inverse,
    rank_and_basis,
    verify_solution,
)

def fmt_obj(obj):
    if isinstance(obj, bool):
        return obj
    if isinstance(obj, int):
        return f"{obj:.4f}"
    if isinstance(obj, float):
        value = 0.0 if abs(obj) <= 1e-12 else obj
        return f"{value:.4f}"
    if isinstance(obj, list):
        return [fmt_obj(item) for item in obj]
    if isinstance(obj, tuple):
        return tuple(fmt_obj(item) for item in obj)
    if isinstance(obj, dict):
        return {key: fmt_obj(value) for key, value in obj.items()}
    return obj

# Part 1 Demo - Phép khử Gauss và các ứng dụng

Notebook này minh họa 3 nhóm trường hợp chính:
1. Hệ vuông khả nghịch với pivot rất nhỏ.
2. Hệ có vô số nghiệm.
3. Hệ vô nghiệm.

Ngoài ra ta còn minh họa các hàm sau:
- back substitution
- determinant
- inverse
- rank and basis
- verify kết quả bằng NumPy

## Case 1 - Hệ vuông khả nghịch

case này sử dụng các hàm sau:
- gaussian_eliminate
- back_substitution
- determinant
- inverse
- verify_solution

Kết quả:
- `swap_count > 0` cho thấy đã có đổi dòng
- `solution_info["type"]` = `unique`
- `residual_norm` thể hiện độ uy tín của nghiệm

In [6]:
A1 = [
    [0.00001, 2.34567, -1.23456],
    [3.5, -4.2, 1.125],
    [2.25, 1.33333, 5.75],
]
b1 = [1.2345, -2.3456, 3.4567]

ref1, info1, swap1 = gaussian_eliminate(A1, b1)

n1 = len(A1)
U1 = [row[:n1] for row in ref1[:n1]]
c1 = [row[n1] for row in ref1[:n1]]

x1_back = back_substitution(U1, c1)
det1 = determinant(A1)
inv1 = inverse(A1)
ver1 = verify_solution(A1, info1, b1)

print("REF augmented =", fmt_obj(ref1))
print("solution_info =", fmt_obj(info1))
print("swap_count =", swap1)
print("x from back_substitution =", fmt_obj(x1_back))
print("det(A1) =", fmt_obj(det1))
print("A1^-1 =", fmt_obj(inv1))
print("verification =", fmt_obj(ver1))

REF augmented = [['3.5000', '-4.2000', '1.1250', '-2.3456'], ['0.0000', '4.0333', '5.0268', '4.9646'], ['0.0000', '0.0000', '-4.1580', '-1.6528']]
solution_info = {'type': 'unique', 'x': ['0.0847', '0.7355', '0.3975'], 'pivot_columns': ['0.0000', '1.0000', '2.0000']}
swap_count = 2
x from back_substitution = ['0.0847', '0.7355', '0.3975']
det(A1) = -58.6972
A1^-1 = [['0.4370', '0.2578', '0.0434'], ['0.2997', '-0.0473', '0.0736'], ['-0.2405', '-0.0899', '0.1399']]
verification = {'is_close': True, 'residual_norm': '0.0000', 'relative_residual': '0.0000', 'numpy_solution': ['0.0847', '0.7355', '0.3975']}


**Nhận xét:** 
- Hệ có nghiệm duy nhất nên `solution_info["type"]` là `unique`.
- Việc `swap_count` lớn hơn 0 cho thấy partial pivoting đã được kích hoạt.
- `back_substitution` cho nghiệm trùng với nghiệm trong `solution_info`.
- `det(A1) ≠ 0` nên ma trận khả nghịch và hàm `inverse()` trả về được ma trận khả nghịch của `A1`.
- `residual_norm` và `relative_residual` rất nhỏ nên nghiệm có độ tin cậy cao.

## Case 2 - Hệ có vô số nghiệm

Case này dùng ma trận phụ thuộc tuyến tính để kiểm tra khả năng:
- phát hiện hệ có vô số nghiệm
- dựng nghiệm riêng
- dựng cơ sở không gian nghiệm
- tính rank và basis

In [7]:
A2 = [
    [1.2, 2.4, -0.6, 3.0],
    [2.4, 4.8, -1.2, 6.0],
    [0.6, 1.2, -0.3, 1.5],
]
b2 = [3.6, 7.2, 1.8]

ref2, info2, swap2 = gaussian_eliminate(A2, b2)
rb2 = rank_and_basis(A2)
ver2 = verify_solution(A2, info2, b2)

print("REF augmented =", fmt_obj(ref2))
print("solution_info =", fmt_obj(info2))
print("swap_count =", swap2)
print("rank_and_basis =", fmt_obj(rb2))
print("verification =", fmt_obj(ver2))

REF augmented = [['2.4000', '4.8000', '-1.2000', '6.0000', '7.2000'], ['0.0000', '0.0000', '0.0000', '0.0000', '0.0000'], ['0.0000', '0.0000', '0.0000', '0.0000', '0.0000']]
solution_info = {'type': 'infinite', 'particular': ['3.0000', '0.0000', '0.0000', '0.0000'], 'nullspace_basis': [['-2.0000', '1.0000', '0.0000', '0.0000'], ['0.5000', '0.0000', '1.0000', '0.0000'], ['-2.5000', '0.0000', '0.0000', '1.0000']], 'pivot_columns': ['0.0000'], 'free_columns': ['1.0000', '2.0000', '3.0000'], 'parameters': ['t1', 't2', 't3'], 'general_form': 'x = [3.0, 0.0, 0.0, 0.0] + t1*[-2.0, 1.0, 0.0, 0.0] + t2*[0.5, 0.0, 1.0, 0.0] + t3*[-2.5, 0.0, 0.0, 1.0]', 'rref_augmented': [['1.0000', '2.0000', '-0.5000', '2.5000', '3.0000'], ['0.0000', '0.0000', '0.0000', '0.0000', '0.0000'], ['0.0000', '0.0000', '0.0000', '0.0000', '0.0000']]}
swap_count = 1
rank_and_basis = {'rank': '1.0000', 'pivot_columns': ['0.0000'], 'free_columns': ['1.0000', '2.0000', '3.0000'], 'rref': [['1.0000', '2.0000', '-0.5000', '2.

**Nhận xét:**  
- Số pivot nhỏ hơn số ẩn nên hệ có vô số nghiệm.
- `solution_info` cho biết một nghiệm riêng và cơ sở của không gian nghiệm.
- `rank_and_basis()` cho thấy hạng của ma trận nhỏ hơn số cột.

## Case 3 - Hệ vô nghiệm

Case này dùng để kiểm tra sự không tồn tại nghiệm của hệ sau khi khử Gauss.

In [8]:
A3 = [
    [1, 1, 1],
    [2, 2, 2],
    [1, -1, 0],
]
b3 = [3, 7, 1]

ref3, info3, swap3 = gaussian_eliminate(A3, b3)

print("REF augmented =", fmt_obj(ref3))
print("solution_info =", fmt_obj(info3))
print("swap_count =", swap3)

REF augmented = [['2.0000', '2.0000', '2.0000', '7.0000'], ['0.0000', '-2.0000', '-1.0000', '-2.5000'], ['0.0000', '0.0000', '0.0000', '-0.5000']]
solution_info = {'type': 'inconsistent', 'pivot_columns': ['0.0000', '1.0000'], 'message': 'The system is inconsistent, so it has no solution.'}
swap_count = 2


**Nhận xét:**  
- Sau khi khử, xuất hiện hàng có dạng `[0, 0, 0 | c]` với `c ≠ 0`.  
- Vì vậy hệ được phân loại là `inconsistent`.  
- Vì vậy không thể dùng `back_substitution`, `inverse` hay `determinant` như trong case 1.